# 📊 So sánh 4 model — CNN vs Transformer (công bằng)

Chấm điểm **cả 4 model trên CÙNG tập valid, CÙNG thước đo** (supervision mAP) → bảng + biểu đồ cho báo cáo IS54A.

| Model | Loại | Epochs |
|-------|------|:---:|
| YOLO11s | CNN | 60 |
| YOLO26s | CNN | 60 |
| RT-DETR-L | Transformer | 10 |
| RF-DETR-S | Transformer | 10 |

**Cần:** 4 file weights đã lưu Drive (`yolo11s_best.pt, yolo26s_best.pt, rtdetr_best.pt, rfdetr_small_best.pth`) + `summary_rfdetr.json`.

`Runtime → T4 GPU → Run all`

In [ ]:
# 1 — Cài đặt + mount Drive + nạp model
!nvidia-smi -L
%pip install -q -U "ultralytics>=8.4.0" rfdetr supervision roboflow
import os, json
from pathlib import Path
import numpy as np
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from ultralytics import YOLO, RTDETR

from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/DrowsyDriver_Results')

UNIFIED = ['close_eyeL','close_eyeR','no_yawn','open_eyeL','open_eyeR','yawn']
UNI_IDX = {n:i for i,n in enumerate(UNIFIED)}

# Nạp model có sẵn → list (tên, loại, epochs, params_M, predictor)
MODELS = []
if (OUT/'yolo11s_best.pt').exists():
    MODELS.append(['YOLO11s','CNN',60,9.4, YOLO(str(OUT/'yolo11s_best.pt')), 'ultra'])
if (OUT/'yolo26s_best.pt').exists():
    MODELS.append(['YOLO26s','CNN',60,9.5, YOLO(str(OUT/'yolo26s_best.pt')), 'ultra'])
if (OUT/'rtdetr_best.pt').exists():
    MODELS.append(['RT-DETR-L','Transformer',10,32.0, RTDETR(str(OUT/'rtdetr_best.pt')), 'ultra'])
if (OUT/'rfdetr_small_best.pth').exists():
    from rfdetr import RFDETRSmall
    MODELS.append(['RF-DETR-S','Transformer',10,30.0, RFDETRSmall(pretrain_weights=str(OUT/'rfdetr_small_best.pth')), 'rfdetr'])
print('Sẽ chấm:', [m[0] for m in MODELS])

RFDETR_NAMES = json.loads((OUT/'summary_rfdetr.json').read_text()).get('classes', []) if (OUT/'summary_rfdetr.json').exists() else []
print('RFDETR_NAMES:', RFDETR_NAMES)

In [ ]:
# 2 — Tải dataset + dựng ground-truth từ tập valid (định dạng YOLO)
from roboflow import Roboflow
rf = Roboflow(api_key='qI3lEKlNpIZpNENdk3MH')
ds = None
for proj in ['datio_yolo','driver-yawn','driver-yawn-wh6wj']:
    try: ds = rf.workspace('nguyen-tuan-dat').project(proj).version(1).download('yolov11'); break
    except Exception: pass
base = Path(ds.location)

# split để chấm: valid (có cả ảnh + nhãn). Nếu không có valid → test
split = 'valid' if (base/'valid'/'images').exists() else 'test'
gt = sv.DetectionDataset.from_yolo(
    images_directory_path=str(base/split/'images'),
    annotations_directory_path=str(base/split/'labels'),
    data_yaml_path=str(base/'data.yaml'))
print(f'  Chấm trên split "{split}": {len(gt)} ảnh')

In [ ]:
# 3 — Hàm predict đưa MỌI model về cùng không gian class UNIFIED
def to_unified_ultra(model, path):
    r = model.predict(path, conf=0.01, verbose=False)[0]
    det = sv.Detections.from_ultralytics(r)
    if len(det):
        det.class_id = np.array([UNI_IDX.get(model.names[int(c)], -1) for c in det.class_id])
        det = det[det.class_id >= 0]
    return det

def to_unified_rfdetr(model, path):
    det = model.predict(path, threshold=0.01)
    if len(det):
        ids = [UNI_IDX.get(RFDETR_NAMES[int(c)] if int(c) < len(RFDETR_NAMES) else '', -1) for c in det.class_id]
        det.class_id = np.array(ids)
        det = det[det.class_id >= 0]
    return det

def predict(model, kind, path):
    return to_unified_rfdetr(model, path) if kind=='rfdetr' else to_unified_ultra(model, path)

In [ ]:
# 4 — Chấm mAP từng model trên cùng tập valid
results = []
for name, kind_lbl, ep, params, model, kind in MODELS:
    print(f'  Đang chấm {name}...')
    preds, targs = [], []
    for path, _img, ann in gt:
        preds.append(predict(model, kind, path))
        targs.append(ann)
    r = MeanAveragePrecision().update(preds, targs).compute()
    results.append({'name':name,'type':kind_lbl,'epochs':ep,'params':params,
                    'mAP50':float(r.map50),'mAP50_95':float(r.map50_95),'mAP75':float(r.map75)})
    print(f'     mAP50={r.map50*100:.2f}%  mAP50-95={r.map50_95*100:.2f}%')

(OUT/'comparison_results.json').write_text(json.dumps(results, indent=2))
print('\n✅  Lưu comparison_results.json')

In [ ]:
# 5 — Bảng + biểu đồ cho báo cáo
import matplotlib.pyplot as plt

print('╔'+'═'*66+'╗')
print('║  BẢNG SO SÁNH — DROWSY DRIVER 6-class (cùng tập valid)            ║')
print('╠'+'═'*66+'╣')
print(f'║  {"Model":<11}{"Loại":<13}{"Epochs":>7}{"Params":>9}{"mAP50":>9}{"mAP50-95":>10} ║')
print('╠'+'─'*66+'╣')
for r in sorted(results, key=lambda x:-x['mAP50_95']):
    print(f'║  {r["name"]:<11}{r["type"]:<13}{r["epochs"]:>7}{r["params"]:>7.1f}M{r["mAP50"]*100:>8.2f}%{r["mAP50_95"]*100:>9.2f}% ║')
print('╚'+'═'*66+'╝')

names  = [r['name'] for r in results]
m50    = [r['mAP50']*100 for r in results]
m5095  = [r['mAP50_95']*100 for r in results]
colors = ['#2196F3' if r['type']=='CNN' else '#FF5722' for r in results]
x = np.arange(len(names)); w = 0.38
fig, ax = plt.subplots(figsize=(11,5))
b1 = ax.bar(x-w/2, m50,   w, label='mAP50',    color=colors, alpha=0.9)
b2 = ax.bar(x+w/2, m5095, w, label='mAP50-95', color=colors, alpha=0.5)
ax.bar_label(b1, fmt='%.1f'); ax.bar_label(b2, fmt='%.1f')
ax.set_xticks(x); ax.set_xticklabels([f'{r["name"]}\n({r["type"]}, {r["epochs"]}ep)' for r in results], fontsize=9)
ax.set_ylabel('mAP (%)'); ax.set_ylim(0,105); ax.legend()
ax.set_title('CNN (xanh) vs Transformer (cam) — Drowsy 6-class', fontweight='bold')
ax.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(OUT/'comparison_chart.png', dpi=130, bbox_inches='tight'); plt.show()
print('✅  Lưu comparison_chart.png vào Drive')

## 📝 Gợi ý nhận xét cho báo cáo
- **CNN (YOLO11/26)** train 60 epoch — nhẹ (~9M params), nhanh, hợp deploy Android.
- **Transformer (RT-DETR/RF-DETR)** chỉ train **10 epoch** mà vẫn cạnh tranh → cho thấy sức mạnh của backbone pretrain (DINOv2), đổi lại nặng (30-32M params) và train chậm hơn.
- Ghi rõ số epoch ở mỗi dòng để so sánh **công bằng** (transformer chưa train hết tiềm năng).
- Tất cả chấm trên **cùng tập valid + cùng supervision mAP** → số liệu so sánh được trực tiếp.